# SinFusion — Test sur OVHCloud JupyterLab
**Projet** : Single-Source Fast Generation (GENAI ING3)

## 1. Setup

In [ ]:
import os, subprocess, sys

# Racine du projet (parent du dossier notebooks/)
PROJECT_ROOT = os.path.dirname(os.path.abspath(''))
os.chdir(PROJECT_ROOT)
print(f"Répertoire : {PROJECT_ROOT}")

In [ ]:
# Lancer le script de setup (installe PyTorch cu118, dépendances, init submodule)
!bash setup.sh

## 2. Préparer l'image source

SinFusion attend les images dans `sinfusion/images/`.  
On télécharge `fruit.png` (fournie par les auteurs dans le repo officiel).

In [ ]:
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt

# ── À CHANGER pour une nouvelle image ────────────────────────────────────────
IMAGE_NAME = 'fruit.png'
# ─────────────────────────────────────────────────────────────────────────────
RUN_NAME   = os.path.splitext(IMAGE_NAME)[0] + '_teacher'
IMAGE_PATH = f'sinfusion/images/{IMAGE_NAME}'

os.makedirs('sinfusion/images', exist_ok=True)

# Téléchargement automatique si c'est une image officielle SinFusion
# Pour une image custom : uploadez-la dans sinfusion/images/ et changez IMAGE_NAME ci-dessus
if not os.path.exists(IMAGE_PATH):
    url = f'https://raw.githubusercontent.com/yanivnik/sinfusion-code/main/images/{IMAGE_NAME}'
    try:
        urllib.request.urlretrieve(url, IMAGE_PATH)
        print(f"Image téléchargée : {IMAGE_PATH}")
    except Exception:
        print(f"Téléchargement échoué — uploadez manuellement '{IMAGE_NAME}' dans sinfusion/images/")
else:
    print(f"Image présente : {IMAGE_PATH}")

img = Image.open(IMAGE_PATH)
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f'Image source : {IMAGE_NAME}  ({img.width}×{img.height} px)')
plt.axis('off')
plt.show()

## 3. Entraînement SinFusion

Entraîne un DDPM sur l'image source.  
Le checkpoint est sauvegardé dans `sinfusion/lightning_logs/fruit.png/checkpoints/last.ckpt`.

> `--diffusion_timesteps 50` = valeur du papier (rapide à tester).  
> Pour un entraînement complet, le code original tourne 50 000 steps — ici on peut réduire via `max_steps` si besoin mais on laisse le défaut du papier.

In [ ]:
%%time
!cd sinfusion && python main.py \
    --task image \
    --image_name {IMAGE_NAME} \
    --run_name {RUN_NAME} \
    --diffusion_timesteps 50 \
    --network_depth 16 \
    --network_filters 64

## 4. TensorBoard — courbe de loss

In [ ]:
%load_ext tensorboard
%tensorboard --logdir sinfusion/lightning_logs --port 6006

## 5. Génération d'échantillons

In [ ]:
%%time
!cd sinfusion && python sample.py \
    --task image \
    --image_name {IMAGE_NAME} \
    --run_name {RUN_NAME} \
    --sample_count 8 \
    --diffusion_timesteps 50 \
    --network_depth 16 \
    --output_dir ../outputs

## 6. Visualisation des résultats

In [ ]:
import glob
import numpy as np

OUTPUT_DIR = f'outputs/{IMAGE_NAME}/{RUN_NAME}'

sample_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.png'))
print(f"{len(sample_files)} fichier(s) trouvé(s) dans {OUTPUT_DIR}")

if not sample_files:
    print("Aucun échantillon trouvé — vérifier que la cellule 5 s'est bien exécutée.")
else:
    n = len(sample_files)
    n_cols = min(4, n)
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(1 + n_rows, n_cols, figsize=(4 * n_cols, 4 * (1 + n_rows)))
    axes = np.array(axes).reshape(1 + n_rows, n_cols)

    axes[0, 0].imshow(img)
    axes[0, 0].set_title('Source', fontweight='bold')
    axes[0, 0].axis('off')
    for j in range(1, n_cols):
        axes[0, j].axis('off')

    for i, path in enumerate(sample_files):
        r, c = divmod(i, n_cols)
        axes[r + 1, c].imshow(Image.open(path))
        axes[r + 1, c].set_title(f'Sample {i}', fontsize=9)
        axes[r + 1, c].axis('off')
    for i in range(n, n_rows * n_cols):
        r, c = divmod(i, n_cols)
        axes[r + 1, c].axis('off')

    fig.suptitle(f'SinFusion — Échantillons générés ({IMAGE_NAME})', fontsize=13)
    plt.tight_layout()
    plt.show()

## 7. Copie du checkpoint teacher\n\nOn copie le checkpoint hors du submodule `sinfusion/` vers `teacher_checkpoint/{IMAGE_NAME}/` pour qu'il soit tracké par Git LFS et accessible aux notebooks suivants.

In [ ]:
import shutil

src = os.path.join(SINFUSION_DIR if 'SINFUSION_DIR' in dir() else os.path.join(PROJECT_ROOT, 'sinfusion'),
                   'lightning_logs', IMAGE_NAME, RUN_NAME, 'checkpoints', 'last.ckpt')
dst_dir = os.path.join(PROJECT_ROOT, 'teacher_checkpoint', IMAGE_NAME)
dst     = os.path.join(dst_dir, 'last.ckpt')

os.makedirs(dst_dir, exist_ok=True)

assert os.path.exists(src), f"Checkpoint source introuvable : {src}\nL'entraînement s'est-il bien terminé ?"
shutil.copy2(src, dst)
print(f"Checkpoint copié : {dst}")
print(f"Taille           : {os.path.getsize(dst) / 1e6:.1f} MB")
print()
print("Prochaine étape : git add + push ce checkpoint, puis lancer le notebook 02.")